In [4]:
!pip -q install faiss-cpu sentence-transformers transformers accelerate

In [5]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [7]:
print("Creating Knowledge Base...")
kb=[]
for _,row in train.iterrows():
    kb.append(str(row[row['answer']]))
print("Knowledge Base Size:",len(kb))

Creating Knowledge Base...
Knowledge Base Size: 2000


In [8]:
embedder=SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings=embedder.encode(kb,convert_to_numpy=True,show_progress_bar=True)
index=faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("FAISS Index Ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

FAISS Index Ready


In [9]:
zs=pipeline('zero-shot-classification',model='facebook/bart-large-mnli')
cross_encoder=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
tokenizer=AutoTokenizer.from_pretrained('bert-base-uncased')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
def retrieve(prompt,k=5):
    emb=embedder.encode([prompt],convert_to_numpy=True)
    D,I=index.search(emb,k)
    return I[0],D[0]

def map3(actual,pred):
    for i,p in enumerate(pred):
        if p==actual:
            return 1/(i+1)
    return 0

## Q1

In [11]:
row=train.iloc[150]
prompt=row["prompt"]
labels=[str(row[c]) for c in ["A","B","C","D","E"]]
correct=str(row[row["answer"]])
res=zs(prompt,candidate_labels=labels)
print(round(res["scores"][res["labels"].index(correct)],3))

0.384


## Q2

In [12]:
retrieved,_=retrieve(prompt,10)
rank=next((i+1 for i,x in enumerate(retrieved) if x==150),None)
print(rank)

10


## Q3

In [13]:
docs=[kb[i] for i in retrieved]
pairs=[[prompt,d] for d in docs]
scores=cross_encoder.predict(pairs)
order=np.argsort(scores)[::-1]
reranked=retrieved[order]
rank=next((i+1 for i,x in enumerate(reranked) if x==150),None)
print(rank)

1


## Q4

In [14]:
row=train.iloc[42]
retrieved,_=retrieve(row["prompt"],5)
context=" ".join(kb[i] for i in retrieved)
text=f"Context: {context} Question: {row['prompt']}"
tokens=tokenizer(text,truncation=False)
print(len(tokens["input_ids"]))

216


## Q5

In [15]:
row=train.iloc[150]
true_doc=kb[150]
prompt=row["prompt"]
labels=[str(row[c]) for c in ["A","B","C","D","E"]]
correct=str(row[row["answer"]])
rag=f"Context: {true_doc} Question: {prompt}"
res=zs(rag,candidate_labels=labels)
print(round(res["scores"][res["labels"].index(correct)],3))

0.989


## Q6

In [16]:
bad_doc=kb[999]
rag=f"Context: {bad_doc} Question: {prompt}"
res=zs(rag,candidate_labels=labels)
print(round(res["scores"][res["labels"].index(correct)],3))

0.529


## Q7

In [17]:
hits=0
for i in range(100):
    row=train.iloc[i]
    ans=str(row[row["answer"]])
    retrieved,_=retrieve(row["prompt"],5)
    if any(ans in kb[j] for j in retrieved):
        hits+=1
print(round(hits,1))

73


## Q8

In [18]:
scores=[]
letters=["A","B","C","D","E"]
for i in range(20):
    row=train.iloc[i]
    prompt=row["prompt"]
    labels=[str(row[c]) for c in letters]
    retrieved,_=retrieve(prompt,5)
    docs=[kb[j] for j in retrieved]
    ce=cross_encoder.predict([[prompt,d] for d in docs])
    best=docs[np.argmax(ce)]
    rag=f"Context: {best} Question: {prompt}"
    res=zs(rag,candidate_labels=labels)
    ranking=np.argsort(res["scores"])[::-1]
    pred=[letters[j] for j in ranking[:3]]
    scores.append(map3(row["answer"],pred))
print(round(np.mean(scores),3))

0.45
